# Hansard RAG — ingestion

Fetches UK parliamentary debates from the Hansard API and transforms them into chunked records ready for indexing.

1. **Discover** debates for a date range → `data/raw/debates_index.json`
2. **Fetch** full content per debate → `data/raw/debates/{ext_id}.json` (idempotent — skips files already on disk)
3. **Transform** raw dumps → cleaned, chunked `data/processed/chunks.jsonl`

In [ ]:
import json
import re
import time
from pathlib import Path

import requests

BASE_URL = "https://hansard-api.parliament.uk"

RAW_DIR = Path("data/raw")
DEBATES_DIR = RAW_DIR / "debates"
PROCESSED_DIR = Path("data/processed")

# make the dir if they dont exist
for d in (DEBATES_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

Small subse, one week of debates for the house of common chambers

In [ ]:
START_DATE = "2026-07-08"
END_DATE = "2026-07-16"
HOUSES = ["Commons"]

# CHUNK_STRATEGY = "paragraphs"
CHUNK_STRATEGY = "contribution"
TARGET_TOKENS = 350
MIN_WORDS = 5 # exclude minor comments

## Stage 1 — discover debates

`search/debates.json` is the discovery endpoint; each result carries a `DebateSectionExtId` 

In [ ]:
PAGE_SIZE = 50


def fetch_debates_page(start_date, end_date, house, skip):
    params = {
        "queryParameters.startDate": start_date,
        "queryParameters.endDate": end_date,
        "queryParameters.house": house,
        "queryParameters.take": PAGE_SIZE,
        "queryParameters.skip": skip,
    }
    resp = requests.get(f"{BASE_URL}/search/debates.json", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def fetch_all_debates(start_date, end_date, house):
    results = []
    skip = 0
    while True:
        page = fetch_debates_page(start_date, end_date, house, skip)
        page_results = page.get("Results", [])
        if not page_results:
            break
        results.extend(page_results)
        print(f"  {house}: fetched {len(results)} so far (TotalResultCount={page.get('TotalResultCount')})")
        if len(page_results) < PAGE_SIZE:
            break
        skip += PAGE_SIZE
        time.sleep(0.5)
    return results

In [ ]:
all_results = []
for house in HOUSES:
    print(f"Searching {house} debates {START_DATE} -> {END_DATE}")
    all_results.extend(fetch_all_debates(START_DATE, END_DATE, house))

# Deduplicate on ExtId
seen = set()
debates_index = []
for r in all_results:
    ext_id = r.get("DebateSectionExtId")
    if ext_id and ext_id not in seen:
        seen.add(ext_id)
        debates_index.append(r)

index_path = RAW_DIR / "debates_index.json"
index_path.write_text(json.dumps(debates_index, indent=2))
print(f"{len(debates_index)} debates ({len(all_results) - len(debates_index)} duplicates removed) -> {index_path}")

debates_index[:3]

## Stage 2 — fetch full debate content

One request per debate, raw JSON dumped to disk keyed by ExtId. Re-running the cell only fetches what's missing (set `FORCE = True` to re-fetch everything).

In [ ]:
FORCE = False


def fetch_debate(ext_id):
    resp = requests.get(f"{BASE_URL}/debates/debate/{ext_id}.json", timeout=60)
    resp.raise_for_status()
    return resp.json()  # API returns JSON null for unknown ids -> None


fetched, skipped, empty = 0, 0, 0
for entry in debates_index:
    ext_id = entry["DebateSectionExtId"]
    out_path = DEBATES_DIR / f"{ext_id}.json"
    if out_path.exists() and not FORCE:
        skipped += 1
        continue

    debate = fetch_debate(ext_id)
    if debate is None:
        print(f"  WARN: null response for {ext_id} ({entry.get('Title', '?').strip()})")
        empty += 1
        continue

    out_path.write_text(json.dumps(debate))
    fetched += 1
    print(f"  fetched {ext_id}  {entry.get('Title', '?').strip()}")
    time.sleep(0.5)  # be polite

print(f"Done: {fetched} fetched, {skipped} skipped (already on disk), {empty} empty")

## Stage 3 — transform to chunks

- `ItemType == "Contribution"` filters out timestamps/procedural items for free
- column-number `<span>` markers embedded in the text are stripped
- speaker/party/constituency parsed from `AttributedTo` ("Bob Blackman (Harrow East) (Con)"); role-holders (Deputy Speaker etc.) keep the full string with null party
- two chunking strategies, for the retrieval experiment later:
  - `contribution` — one chunk per contribution (naive baseline)
  - `paragraphs` — group paragraphs up to `TARGET_TOKENS`

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
SPEAKER_RE = re.compile(r"^(?P<speaker>[^(]+?)\s*\((?P<constituency>[^)]+)\)\s*\((?P<party>[^)]+)\)\s*$")


def clean_text(value):
    text = TAG_RE.sub("", value)
    text = text.replace("\u00a0", " ")
    return text.strip()


def approx_tokens(text):
    return int(len(text.split()) * 1.3)


def parse_speaker(attributed_to):
    m = SPEAKER_RE.match(attributed_to or "")
    if m:
        return {
            "speaker": m["speaker"].strip(),
            "constituency": m["constituency"].strip(),
            "party": m["party"].strip(),
        }
    return {"speaker": (attributed_to or "").strip(), "constituency": None, "party": None}


def split_paragraphs(text):
    paragraphs = re.split(r"(?:\r\n|\n){2,}", text)
    return [p.strip() for p in paragraphs if p.strip()]


def group_paragraphs(paragraphs, target_tokens=TARGET_TOKENS):
    """Greedily group consecutive paragraphs up to a token budget."""
    chunks, current, current_tokens = [], [], 0
    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if current and current_tokens + para_tokens > target_tokens:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens
    if current:
        chunks.append("\n\n".join(current))
    return chunks


def chunk_contribution(text, strategy):
    if strategy == "contribution":
        return [text]
    return group_paragraphs(split_paragraphs(text))


def hansard_url(house, date, ext_id):
    return f"https://hansard.parliament.uk/{house}/{date[:10]}/debates/{ext_id}"


def transform_debate(raw, strategy):
    overview = raw.get("Overview") or {}
    records = []
    for item in raw.get("Items") or []:
        if item.get("ItemType") != "Contribution":
            continue
        text = clean_text(item.get("Value") or "")
        if len(text.split()) < MIN_WORDS:
            continue

        speaker_info = parse_speaker(item.get("AttributedTo"))
        chunks = chunk_contribution(text, strategy)
        for idx, chunk_text in enumerate(chunks):
            records.append({
                "chunk_id": f"{item.get('ExternalId') or item.get('ItemId')}_{idx}",
                "debate_ext_id": overview.get("ExtId"),
                "debate_title": (overview.get("Title") or "").strip(),
                "sitting_date": (overview.get("Date") or "")[:10],
                "house": overview.get("House"),
                "location": overview.get("Location"),
                "member_id": item.get("MemberId"),
                **speaker_info,
                "order_in_section": item.get("OrderInSection"),
                "chunk_index": idx,
                "n_chunks": len(chunks),
                "text": chunk_text,
                "hansard_url": hansard_url(
                    overview.get("House", "Commons"),
                    overview.get("Date") or "",
                    overview.get("ExtId") or "",
                ),
            })
    return records

In [ ]:
# out_path = PROCESSED_DIR / "chunks.jsonl"
out_path = PROCESSED_DIR / "chunks_contribution.jsonl"

EXCLUDE_TITLES = {"House of Commons", "Commons Chamber", "Business Before Questions"}

n_debates, n_chunks, n_excluded = 0, 0, 0
with out_path.open("w") as f:
    for path in sorted(DEBATES_DIR.glob("*.json")):
        raw = json.loads(path.read_text())
        title = ((raw.get("Overview") or {}).get("Title") or "").strip()
        if title in EXCLUDE_TITLES:
            n_excluded += 1
            continue
        records = transform_debate(raw, CHUNK_STRATEGY)
        for record in records:
            f.write(json.dumps(record) + "\n")
        n_debates += 1
        n_chunks += len(records)

print(f"Processed {n_debates} debates ({n_excluded} excluded) -> {n_chunks} chunks ({CHUNK_STRATEGY}) -> {out_path}")

## Sanity checks

Eyeball a few records and basic stats before moving on to indexing.

In [ ]:
chunks = [json.loads(line) for line in out_path.open()]

print(f"total chunks:        {len(chunks)}")
print(f"debates represented: {len({c['debate_ext_id'] for c in chunks})}")
print(f"speakers parsed:     {sum(1 for c in chunks if c['party'])} with party / {len(chunks)} total")
print(f"multi-chunk contributions: {sum(1 for c in chunks if c['n_chunks'] > 1 and c['chunk_index'] == 0)}")

lengths = sorted(approx_tokens(c["text"]) for c in chunks)
print(f"chunk tokens (approx): min={lengths[0]}, median={lengths[len(lengths)//2]}, max={lengths[-1]}")

chunks[1]